# Prompt Guard — Regex Baseline

This baseline detects structured sensitive information in a prompt and returns a policy decision
(**allow / warn / block**) and the location of each match. The detector uses hand-written regular
expressions and format checks. There is no model training.

1. **Find candidates** in the raw prompt using regular expressions.
2. **Check the format** where it helps distinguish sensitive data from ordinary numbers.
3. **Apply the shared policy** from config/policy.yaml. The most severe match determines the decision.

| Rule | Examples of supported formats | Dataset label |
|---|---|---|
| E-mail | anna.schmidt@example.com, alex+work@example.org | EMAIL |
| Phone | +49 7131 123456, 07131/123456, (202) 555-0148 | PHONENUMBER |
| IBAN | Compact or groups of four, with a valid checksum | IBAN |
| Credit card | 13–19 digits, compact or grouped, with a valid checksum | CREDITCARDNUMBER |
| Domestic account number | 5–17 consecutive digits following an account-number keyword | ACCOUNTNUMBER |

Open this notebook from the project's notebooks/ folder and select the project Python environment.
The synthetic examples run with the policy alone. The validation and MLflow sections also need the
local dataset files and the committed data/splits.csv. The validation split and metrics match the
TF-IDF + Logistic Regression notebook. The test split is reserved for the team's final comparison.

## 1. Setup and shared policy

The project helper supplies the label mapping and the decision function. The notebook can be
started with the project root or the notebooks directory as the working directory.

In [1]:
import re                     # regular expressions: the core of this baseline
import sys                    # to make the project's src/ folder importable
from hashlib import sha256    # to compute a fingerprint (hash) of the policy file
from pathlib import Path      # platform-independent file paths

from IPython.display import display   # shows DataFrames as nice tables in the notebook
import pandas as pd
import yaml                   # to read config/policy.yaml

# ----------------------------------------------------------------------------- project folder
# The notebook can be started from the project root OR from notebooks/. To find the
# project folder in both cases, we start at the current working directory and walk
# up through its parent folders until we find the one that contains src/data_utils.py.
# next(..., None) returns the first folder that fulfils the condition, or None.
working_dir = Path.cwd().resolve()
PROJECT_DIR = next(
    (folder for folder in (working_dir, *working_dir.parents)
     if (folder / "src" / "data_utils.py").is_file()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Open AIOps_PromptGuard in VS Code and run this notebook from its notebooks/ folder."
    )

# Put src/ at the front of Python's search path so that "import data_utils" works.
# data_utils contains the helpers that ALL models share (same split, same policy),
# which is what makes the comparison between the models fair.
sys.path.insert(0, str(PROJECT_DIR / "src"))
from data_utils import load_policy, load_split, prompt_decision, split_fingerprint

# ----------------------------------------------------------------------------- constants
CLASSES = ["allow", "warn", "block"]   # the three prompt decisions, least severe first
SEED = 42                              # same random seed as in the TF-IDF notebook
RULESET_VERSION = "1.0"                # increase this whenever the regex rules change

# ----------------------------------------------------------------------------- policy
# load_policy() returns the policy prepared for fast lookups, e.g.
#   policy["label_to_decision"]["IBAN"]  -> "block"
#   policy["label_to_decision"]["EMAIL"] -> "warn"
policy = load_policy()

# We also read the raw bytes of the policy file. Its SHA-256 hash identifies the
# exact policy content, so the MLflow run later shows which policy was used.
policy_bytes = (PROJECT_DIR / "config" / "policy.yaml").read_bytes()
policy_file = yaml.safe_load(policy_bytes)
POLICY_SHA256 = sha256(policy_bytes).hexdigest()

# Use the version written in the YAML file ("version: 0.1"). If the file has no
# version entry, fall back to the first 12 characters of the hash.
POLICY_VERSION = (
    policy_file.get("version") or policy_file.get("policy_version")
    or f"sha256:{POLICY_SHA256[:12]}"
)
print(f"Policy version: {POLICY_VERSION}")

Policy version: 0.1


## 2. Define the detection rules

Phone candidates need a recognisable phone format or a nearby phone keyword. An account
number needs an explicit account keyword because its digits can also look like an order ID.
The IBAN rule covers DE, FR, IT and six other common European formats. IBAN and card
checksums help reject ordinary strings with a similar shape.

The policy gets the exact detected label and the original character offsets. The end offset
is exclusive, so prompt[start:end] gives the matched value.

In [2]:
# =============================================================================
# REGEX CHEAT SHEET (the syntax used in this cell)
# -----------------------------------------------------------------------------
#   r"..."          raw string: a backslash stays a backslash (needed for regex)
#   rf"..."         raw f-string: {NAME} inserts a Python variable into the pattern,
#                   {{ and }} produce literal regex braces, e.g. [0-9]{{4}} -> [0-9]{4}
#   [0-9]           one digit             [A-Za-z]   one ASCII letter
#   \w              letter, digit or _    \b         word boundary
#   \s / \S         whitespace / NOT whitespace
#   [^\S\r\n]       "whitespace, but not a line break" (NOT non-whitespace, NOT \r, NOT \n)
#   x?  x*  x+      x optional / 0 or more / 1 or more
#   x{5,16}         x repeated 5 to 16 times
#   (?:...)         group without capturing (only for structure)
#   (?P<name>...)   named group: its text can later be read as match["name"]
#   (?<!...)        negative lookbehind: the text BEFORE must not match ...
#   (?!...)         negative lookahead:  the text AFTER  must not match ...
#   (?=...)         positive lookahead:  the text AFTER  must match ...
#   (?ai:...)       inside this group: ASCII-only matching + ignore upper/lower case
#   a|b             a or b
# Lookarounds only check the neighbourhood. They do not become part of the match,
# so the reported value contains only the PII itself.
# =============================================================================

# ----------------------------------------------------------------------------- shared building blocks
# Horizontal whitespace: normal space, no-break space (U+00A0) and narrow no-break
# space (U+202F). Text copied from PDFs or websites often contains the last two.
HSPACE = r"[   ]"

# Characters allowed between the digit groups of a phone number:
# space, dot, slash, hyphen, e.g. "07131 123456", "07131/123456", "555-0148".
PHONE_SEP = r"[ .  /-]"

# A number must not start in the middle of another token:
#   (?<![\w+])        no letter, digit, "_" or "+" directly before the first digit
#                     -> "ABC12345" or "x12345" are not used as a number start
#   (?<![0-9][./-])   not directly after "1." / "1/" / "1-"
#                     -> we do not start inside dates like "01.12.2026"
NUMBER_START = r"(?<![\w+])(?<![0-9][./-])"

# The same idea at the end of a number. The number must NOT be followed by
#   [\w]              a letter or digit                 ("12345abc")
#   [./-][0-9]        ".5", "/5" or "-5"                ("123.45", part of a date)
#   {HSPACE}[0-9]     a space and another digit         ("1234 5678" would continue)
# Because of this, the regex can only stop at the real end of a number.
NUMBER_END = rf"(?![\w]|[./-][0-9]|{HSPACE}[0-9])"

# Text that may stand between a keyword and its value, for example:
#   "Kontonummer: 123", "Kontonummer lautet 123", "account number is 123",
#   "numéro de compte est 123", "Telefon = 123", "Konto-Nr. #123"
# [^\S\r\n]* = any spaces/tabs, but no line break (the value must be on the same line).
VALUE_PREFIX = r"[^\S\r\n]*(?:(?:lautet|ist|is|est|è)\b[^\S\r\n]*)?[:=#]?[^\S\r\n]*"

# ----------------------------------------------------------------------------- e-mail
# Structure: local part @ domain labels . top-level domain
#   e.g. anna.schmidt@example.com, alex+work@mail.example.org
EMAIL_PATTERN = re.compile(
    # Nothing address-like directly before (prevents matching inside a longer token)
    r"(?<![\w.%+@-])"
    # Local part: blocks of allowed characters, separated by single dots
    # ("anna.schmidt", "alex+work", "o'neil"). Two dots in a row are not allowed.
    r"(?P<value>[A-Za-z0-9_%+'-]+(?:\.[A-Za-z0-9_%+'-]+)*@"
    # Domain labels, each followed by a dot: "mail." "example."
    # A label is 1-63 characters, starts and ends with a letter/digit, "-" only inside.
    r"(?:[A-Za-z0-9](?:[A-Za-z0-9-]{0,61}[A-Za-z0-9])?\.)+"
    # Top-level domain: 2-63 letters ("com", "de", "museum"). Closes the group "value".
    r"[A-Za-z]{2,63})"
    # The address ends here: no further address characters may follow. A dot is only
    # forbidden if more text follows, so a sentence-ending "." is allowed and is NOT
    # part of the match ("Write to anna@example.com." -> "anna@example.com").
    r"(?![\w@-]|\.[\w-])"
)

# ----------------------------------------------------------------------------- IBAN
# An IBAN = 2-letter country code + 2 check digits + national account part.
# The total length is fixed per country (source: SWIFT IBAN registry).
# Only the countries listed here are recognised.
IBAN_LENGTHS = {
    "AT": 20, "BE": 16, "CH": 21, "DE": 22, "ES": 24,
    "FR": 27, "GB": 22, "IT": 27, "NL": 18,
}

# Build one exact pattern per country. After "DE89" the rest is written in groups of
# four characters; the space between groups is optional, so both
#   "DE89370400440532013000"  and  "DE89 3704 0044 0532 0130 00"  match.
iban_formats = []
for country, length in IBAN_LENGTHS.items():
    # Example DE: 22 - 4 = 18 characters after "DE89" -> 4 full groups + 2 characters
    full_groups, remainder = divmod(length - 4, 4)
    # (?: optional space + 4 characters ){4}  -> the full groups
    tail = rf"(?:{HSPACE}?[A-Z0-9]{{4}}){{{full_groups}}}"
    if remainder:
        # The last, shorter group (here: optional space + 2 characters)
        tail += rf"{HSPACE}?[A-Z0-9]{{{remainder}}}"
    # Country code + exactly 2 check digits + the tail
    iban_formats.append(rf"{country}[0-9]{{2}}{tail}")

# Combine all countries into one pattern with "|" (= or).
# (?ai:...) makes the group case-insensitive, so "de89 3704 ..." is found as well.
# (?<!\w) and (?!\w): no letters/digits may be glued to the IBAN on either side.
iban_body = "|".join(iban_formats)
IBAN_PATTERN = re.compile(rf"(?<!\w)(?P<value>(?ai:{iban_body}))(?!\w)")

# ----------------------------------------------------------------------------- account number
# A plain domestic account number is just digits, which also look like order numbers,
# customer IDs or invoice numbers. That is why we ONLY report digits that follow an
# explicit account keyword (German, English, French, Italian).
ACCOUNT_PATTERN = re.compile(
    # \b = keyword starts at a word boundary. Supported keywords:
    #   kontonummer | konto-nr. / kontonr | account number / account no.
    #   numéro de compte | numero (di) conto
    r"\b(?:kontonummer|konto-?nr\.?|account\s+(?:number|no\.?)|"
    r"num[ée]ro\s+de\s+compte|numero\s+(?:di\s+)?conto)"
    # Optional ": " / " lautet " / " is " ..., then the value: 5 to 17 digits.
    # The number must end there (no further letter/digit, no ".5" / "-5").
    rf"{VALUE_PREFIX}(?P<value>[0-9]{{5,17}})(?![\w]|[./-][0-9])",
    re.IGNORECASE,   # "Kontonummer", "KONTONUMMER", "kontonummer" all match
)

# ----------------------------------------------------------------------------- credit card
# Card numbers have 13-19 digits. Between any two digits a space or hyphen is allowed,
# so "4111111111111111", "4111 1111 1111 1111" and "4111-1111-1111-1111" all match.
# Structure: first digit + 12 to 18 further digits (each with an optional separator).
# Whether the candidate is really a card number is decided afterwards by the Luhn
# checksum in valid_card(). Note: IMEI numbers of phones (15 digits, written like
# "56-463705-181679-8") also pass the Luhn check and are matched here as well.
CARD_PATTERN = re.compile(
    rf"{NUMBER_START}(?P<value>[0-9](?:[   -]?[0-9]){{12,18}}){NUMBER_END}"
)

# ----------------------------------------------------------------------------- phone number
# Phone keywords in DE / EN / FR / IT. The pattern ends with "$" (end of text) because
# it is NOT searched in the whole prompt, but only in the text directly BEFORE a phone
# candidate (see plausible_phone). "$" therefore means: the keyword (plus an optional
# VALUE_PREFIX such as ": ") must stand right in front of the number,
# e.g. "Telefon: 07131 123456" or "phone number is 555 0148".
PHONE_CONTEXT = re.compile(
    r"\b(?:tel\.?|telefon(?:nummer)?|rufnummer|handy(?:nummer)?|"
    r"phone(?:\s+number)?|mobile(?:\s+number)?|t[ée]l[ée]phone|"
    r"num[ée]ro\s+de\s+t[ée]l[ée]phone|cellulare|telefono|numero\s+di\s+telefono)"
    rf"{VALUE_PREFIX}$",
    re.IGNORECASE,
)

# Phone candidate. The pattern is deliberately broad; plausible_phone() filters it.
PHONE_PATTERN = re.compile(
    # Group "international": optional "+" or "00" (but "00" only if a digit 1-9 follows,
    # so "000..." is no prefix). Example: "+49", "0049".
    rf"{NUMBER_START}(?P<value>(?P<international>\+|00(?=[1-9]))?"
    # Group "main" starts here. Optional part in brackets with an optional number
    # before it: "49 (0)" in "+49 (0)7131 ..." or "(202)" in "(202) 555-0148".
    rf"(?P<main>(?:(?:[0-9]{{1,3}}{PHONE_SEP}?)?\([0-9]{{1,5}}\){PHONE_SEP}?)?"
    # The digits: 6 to 17 digits, any of them separated by space, ".", "/" or "-".
    # Closes the group "main".
    rf"[0-9](?:{PHONE_SEP}?[0-9]){{5,16}})"
    # Optional extension: "x123", " ext. 123". Then the number must end (NUMBER_END).
    rf"(?:{HSPACE}*(?:x|ext\.?){HSPACE}*[0-9]{{1,5}})?){NUMBER_END}",
    re.IGNORECASE,   # "X" / "EXT" for the extension
)

# Dates such as "2026-12-01" or "01.12.2026" have the same shape as short phone
# numbers. Candidates that look exactly like a date are rejected in plausible_phone().
DATE_SHAPE = re.compile(
    r"(?:[0-9]{4}[./-][0-9]{1,2}[./-][0-9]{1,2}|"
    r"[0-9]{1,2}[./-][0-9]{1,2}[./-][0-9]{4})"
)

# North American format (USA/Canada): optional "1 ", 3-digit area code (in brackets or
# followed by a separator), 3-digit exchange, 4-digit line number.
# Area code and exchange never start with 0 or 1 -> [2-9].
# Examples: "(202) 555-0148", "202-555-0148", "1 202 555 0148".
NORTH_AMERICAN_PHONE = re.compile(
    rf"(?:1{PHONE_SEP})?(?:\([2-9][0-9]{{2}}\){PHONE_SEP}?|[2-9][0-9]{{2}}{PHONE_SEP})"
    rf"[2-9][0-9]{{2}}{PHONE_SEP}[0-9]{{4}}"
)

# Matches every character that is NOT a digit. NON_DIGIT.sub("", text) removes them:
# "4111 1111-1111" -> "411111111111"
NON_DIGIT = re.compile(r"[^0-9]")

In [3]:
# =============================================================================
# VALIDATORS
# A regex only checks the SHAPE of a candidate. The functions below check whether the
# candidate is also plausible as real PII. Each receives a regex match object and
# returns True (keep the candidate) or False (discard it).
# =============================================================================

def valid_iban(match):
    """Check the IBAN check digits (ISO 7064, mod 97).

    Every real IBAN fulfils: (rearranged IBAN as one big number) % 97 == 1.
    A random string of the right shape passes this only with a chance of about 1 %,
    so typos and made-up values are rejected.
    """
    # Remove optional spaces and use upper case: "de89 3704 ..." -> "DE893704..."
    iban = re.sub(HSPACE, "", match["value"]).upper()
    # Move country code + check digits to the end: "DE89" + rest -> rest + "DE89"
    rotated = iban[4:] + iban[:4]
    # Replace every letter by a two-digit number: A=10, B=11, ..., Z=35
    # (ord("A") is 65, and 65 - 55 = 10). Digits stay as they are.
    numeric = "".join(char if char.isdigit() else str(ord(char) - 55) for char in rotated)
    # The check digits are always between 02 and 98; then the mod-97 test
    return 2 <= int(iban[2:4]) <= 98 and int(numeric) % 97 == 1


def valid_card(match):
    """Check a card number with the Luhn checksum (used by Visa, Mastercard, Amex, ...).

    Note: most card numbers in the synthetic dataset are random digits and fail this
    check (only about 9 % pass). Real card numbers always pass it.
    """
    # Keep only the digits: "4111 1111 1111 1111" -> "4111111111111111"
    digits = NON_DIGIT.sub("", match["value"])
    # Card numbers never start with 0. Numbers with only one repeated digit
    # ("0000 0000 ...", "1111 1111 ...") are placeholders, not real cards.
    if digits[0] == "0" or len(set(digits)) == 1:
        return False
    # Luhn algorithm: go through the digits from RIGHT to LEFT and double every
    # second digit. If doubling gives a two-digit number (> 9), subtract 9
    # (same as adding its two digits, e.g. 16 -> 1 + 6 = 7 = 16 - 9).
    # The number is valid if the sum of all digits is divisible by 10.
    total = 0
    for index, char in enumerate(reversed(digits)):
        digit = int(char)
        if index % 2:           # index 1, 3, 5, ... = every second digit from the right
            digit *= 2
            if digit > 9:
                digit -= 9
        total += digit
    return total % 10 == 0


def plausible_phone(match):
    """Decide whether a phone candidate is really a phone number.

    Long digit strings are everywhere in prompts (order numbers, IDs, amounts), so a
    candidate is only accepted if it has the right length AND at least one extra hint
    (international prefix, phone keyword, US format or national format).
    """
    main = match["main"]                     # the number without "+"/"00" and extension
    international = match["international"]   # "+", "00" or None
    # Count the digits. In "+49 (0)7131 123456" the "(0)" is not dialled from abroad,
    # so it is removed before counting (only for international numbers).
    digits = NON_DIGIT.sub("", main.replace("(0)", "", 1) if international else main)
    # Phone numbers have at most 15 digits (ITU standard E.164); fewer than 7 is too
    # short. Candidates with exactly the shape of a date are rejected.
    if not 7 <= len(digits) <= 15 or DATE_SHAPE.fullmatch(main):
        return False

    # The (at most) 80 characters directly before the candidate, to look for a keyword
    left_context = match.string[max(0, match.start() - 80):match.start()]
    # Accept the candidate if at least ONE of these hints is true:
    return (
        # 1) international format: "+49 7131 ..." / "0049 7131 ..." (no 0 after the prefix)
        bool(international and digits[0] != "0")
        # 2) a phone keyword stands directly before it: "Telefon: ..."
        or PHONE_CONTEXT.search(left_context) is not None
        # 3) North American format: "(202) 555-0148"
        or NORTH_AMERICAN_PHONE.fullmatch(main) is not None
        # 4) national format with trunk prefix 0 and typical length: "07131 123456"
        or (digits.startswith("0") and 9 <= len(digits) <= 12)
    )


# =============================================================================
# RULES
# Each rule = (dataset label, regex pattern, validator or None).
# The ORDER is the priority: when two matches overlap, the rule listed first keeps
# its match. Example: the IBAN "DE89 3704 0044 0532 0130 00" contains digit groups
# that also look like a phone number. Because IBAN comes before PHONENUMBER, the
# text is reported once as IBAN and not a second time as a phone number.
# =============================================================================
RULES = [
    ("EMAIL", EMAIL_PATTERN, None),
    ("IBAN", IBAN_PATTERN, valid_iban),
    ("ACCOUNTNUMBER", ACCOUNT_PATTERN, None),
    ("CREDITCARDNUMBER", CARD_PATTERN, valid_card),
    ("PHONENUMBER", PHONE_PATTERN, plausible_phone),
]
RULE_LABELS = [label for label, _, _ in RULES]

# Safety check: every label we can predict must exist in the shared policy.
# Otherwise prompt_decision() would silently treat it as "warn".
missing_labels = set(RULE_LABELS) - set(policy["label_to_decision"])
if missing_labels:
    raise ValueError(f"These regex labels are missing from the project policy: {sorted(missing_labels)}")


def detect_pii(prompt):
    """Find all PII matches in one prompt.

    Returns a list of spans in the same format as the dataset's privacy_mask, e.g.
        [{"start": 12, "end": 36, "label": "EMAIL", "value": "anna.schmidt@example.com"}]
    sorted by position. "end" is exclusive, so prompt[start:end] == value.
    """
    spans = []
    for label, pattern, validator in RULES:          # rules in priority order
        for match in pattern.finditer(prompt):       # every candidate of this rule
            # Discard candidates that fail the plausibility check
            if validator is not None and not validator(match):
                continue
            # Position of the group "value" only. For an account number this is the
            # number without the keyword "Kontonummer: " in front of it.
            start, end = match.span("value")
            # Discard the candidate if it overlaps a span that a higher-priority rule
            # has already found. Two ranges overlap if each starts before the other ends.
            if any(start < span["end"] and span["start"] < end for span in spans):
                continue
            spans.append({"start": start, "end": end, "label": label, "value": match["value"]})
    # Rules were applied one after another, so sort the result by position in the text
    return sorted(spans, key=lambda span: span["start"])


def predict_prompt(prompt):
    """Return (decision, spans) for one prompt.

    The decision comes from the shared policy, exactly like the reference labels:
    the most severe label wins (block > warn > allow); no match -> "allow".
    """
    spans = detect_pii(prompt)
    return prompt_decision(spans, policy), spans


# Overview: which decision each regex label leads to under the project policy
display(pd.Series({label: policy["label_to_decision"][label] for label in RULE_LABELS},
                  name="Project policy decision").to_frame())

,Project policy decision
EMAIL,warn
IBAN,block
ACCOUNTNUMBER,block
CREDITCARDNUMBER,block
PHONENUMBER,warn


## 3. Try the baseline on synthetic prompts

These examples check the rules, including one prompt with both an e-mail and an IBAN.
You can call predict_prompt with a new prompt to see its decision and spans.

In [4]:
# Hand-written test prompts: (prompt, expected labels in order of appearance).
# They work like small unit tests: if a rule is changed and breaks one of these cases,
# the assert below stops the notebook with the failing prompt.
examples = [
    # No PII at all -> allow
    ("Fasse bitte unsere neue Urlaubsregelung zusammen.", []),
    # E-mail: normal address and address with "+" tag; the final "." / "!" is not included
    ("Schreibe an anna.schmidt@example.com.", ["EMAIL"]),
    ("Schreib an alex+arbeit@example.org!", ["EMAIL"]),
    # Phone: international with "(0)", national with keyword and "/", US format
    ("Ruf bitte unter +49 (0)7131 123456 an.", ["PHONENUMBER"]),
    ("Telefon: 07131/123456.", ["PHONENUMBER"]),
    ("Please call (202) 555-0148.", ["PHONENUMBER"]),
    # IBAN: compact and in groups of four; two different labels in one prompt
    ("Bitte überweise den Betrag an DE89370400440532013000.", ["IBAN"]),
    ("Sende anna.schmidt@example.com die IBAN DE89 3704 0044 0532 0130 00.", ["EMAIL", "IBAN"]),
    # Account number only after a keyword (German and French)
    ("Kontonummer: 1234567890.", ["ACCOUNTNUMBER"]),
    ("Numéro de compte : 1234567890.", ["ACCOUNTNUMBER"]),
    # Credit card test number in groups of four (passes the Luhn check)
    ("Carta: 4111 1111 1111 1111.", ["CREDITCARDNUMBER"]),
    # Negative cases: year, ticket number and date must NOT be detected ...
    ("Umsatz 2023, Ticket 12345678, Termin 01.12.2026.", []),
    # ... and neither must values with a wrong checksum (last digit changed)
    ("Beispielwerte mit Tippfehlern: DE89370400440532013001 und 4111-1111-1111-1112.", []),
]

example_rows = []
for prompt, expected_labels in examples:
    decision, spans = predict_prompt(prompt)
    # Check 1: exactly the expected labels were found, in the expected order
    assert [span["label"] for span in spans] == expected_labels, (prompt, spans)
    # Check 2: the offsets point exactly to the reported value
    assert all(prompt[span["start"]:span["end"]] == span["value"] for span in spans)
    # Check 3: the decision equals the most severe decision of the expected labels.
    # max(..., key=CLASSES.index) picks the decision that comes last in
    # ["allow", "warn", "block"], i.e. the most severe one. No labels -> "allow".
    expected_decision = max(
        (policy["label_to_decision"][label] for label in expected_labels),
        key=CLASSES.index,
        default="allow",
    )
    assert decision == expected_decision, (prompt, decision, expected_decision)
    # One table row per example ("—" if nothing was found)
    example_rows.append({
        "prompt": prompt,
        "decision": decision,
        "labels": ", ".join(span["label"] for span in spans) or "—",
        "matches": ", ".join(span["value"] for span in spans) or "—",
    })

# Edge case: an empty prompt must not crash and must be allowed
assert predict_prompt("")[0] == "allow"
# Show the results; max_colwidth keeps long prompts readable
with pd.option_context("display.max_colwidth", 90):
    display(pd.DataFrame(example_rows))

,prompt,decision,labels,matches
0,Fasse bitte unsere neue Urlaubsregelung zusammen.,allow,—,—
1,Schreibe an anna.schmidt@example.com.,warn,EMAIL,anna.schmidt@example.com
2,Schreib an alex+arbeit@example.org!,warn,EMAIL,alex+arbeit@example.org
3,Ruf bitte unter +49 (0)7131 123456 an.,warn,PHONENUMBER,+49 (0)7131 123456
4,Telefon: 07131/123456.,warn,PHONENUMBER,07131/123456
5,Please call (202) 555-0148.,warn,PHONENUMBER,(202) 555-0148
6,Bitte überweise den Betrag an DE89370400440532013000.,block,IBAN,DE89370400440532013000
7,Sende anna.schmidt@example.com die IBAN DE89 3704 0044 0532 0130 00.,block,"EMAIL, IBAN","anna.schmidt@example.com, DE89 3704 0044 0532 0130 00"
8,Kontonummer: 1234567890.,block,ACCOUNTNUMBER,1234567890
9,Numéro de compte : 1234567890.,block,ACCOUNTNUMBER,1234567890


## 4. Run on the shared validation split

Prediction reads only source_text. The decision and privacy_mask columns are used after
prediction to calculate metrics. The split fingerprint identifies the exact split file.
For comparison with the saved TF-IDF results, check for fingerprint 6afd6a257de1 and policy 0.1.

In [5]:
import json                       # privacy_mask can be stored as a JSON string
import time                       # to measure the response time later
from collections import Counter   # to count TP / FP / FN per label

import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score

# Fingerprint of data/splits.csv: proves that this run used the same split as the
# other models (the TF-IDF run shows 6afd6a257de1).
SPLIT_FINGERPRINT = split_fingerprint()
# The validation split with all dataset columns plus the reference column "decision",
# which load_split() computes from privacy_mask and the policy.
val = load_split("validation")
print(f"Split fingerprint: {SPLIT_FINGERPRINT} | Policy: {POLICY_VERSION}")
print(f"Validation prompts: {len(val):,}")

# Predict every prompt. Only the raw text (source_text) goes into the detector.
# The annotations are not used here, so there is no information leak.
predictions = [predict_prompt(prompt) for prompt in val["source_text"]]
# Store decision and spans as two new columns next to the reference data.
# assign() returns a new DataFrame and leaves val unchanged.
validation = val.assign(
    predicted_decision=[decision for decision, _ in predictions],
    predicted_spans=[spans for _, spans in predictions],
)

Split fingerprint: 6afd6a257de1 | Policy: 0.1
Validation prompts: 31,380


## 5. Evaluate prompt decisions

**Macro F1** gives allow, warn and block the same weight. **Recall block** measures the
share of true block prompts that were blocked. The report uses the full project policy,
including cases whose PII types are outside the regex rules. The confusion matrix shows
counts with true classes in the rows and predictions in the columns.

In [6]:
def decision_metrics(frame):
    """Macro F1 and recall per class for the prompt decisions in one DataFrame."""
    actual = frame["decision"]               # reference decision from the annotations
    predicted = frame["predicted_decision"]  # decision of the regex baseline
    return {
        # F1 per class, then the plain mean over allow, warn and block.
        # Every class counts the same, even though warn is much more frequent.
        "macro_f1": f1_score(actual, predicted, labels=CLASSES, average="macro", zero_division=0),
        # Recall of ONE class = share of its prompts that got the correct decision.
        # labels=[label] restricts the calculation to this class.
        # recall_block is the most important value: missed block prompts are data leaks.
        **{
            f"recall_{label}": recall_score(
                actual, predicted, labels=[label], average="macro", zero_division=0
            )
            for label in CLASSES
        },
    }


# Metrics for all prompts and separately for each language (de, en, fr, it)
overall = decision_metrics(validation)
by_language = {
    language: {"prompts": len(part), **decision_metrics(part)}
    for language, part in validation.groupby("language")
}
# One table: first the row "overall", then one row per language
decisions = pd.DataFrame.from_dict(
    {"overall": {"prompts": len(validation), **overall}, **by_language},
    orient="index",
)
display(decisions.round(4))

# Precision, recall and F1 for each class
actual = validation["decision"]
predicted = validation["predicted_decision"]
print(classification_report(actual, predicted, labels=CLASSES, digits=3, zero_division=0))

# Confusion matrix: rows = true decision, columns = predicted decision.
# The diagonal holds the correct predictions; e.g. row "block" / column "allow"
# counts the block prompts the baseline let through.
confusion = pd.DataFrame(
    confusion_matrix(actual, predicted, labels=CLASSES),
    index=pd.Index(CLASSES, name="True decision"),
    columns=pd.Index(CLASSES, name="Predicted decision"),
)
display(confusion)

,prompts,macro_f1,recall_allow,recall_warn,recall_block
overall,31380,0.2217,1.0,0.1704,0.1256
de,7922,0.2215,1.0,0.1665,0.1306
en,6522,0.2288,1.0,0.1759,0.1289
fr,9292,0.2212,1.0,0.1720,0.1254
it,7644,0.2165,1.0,0.1678,0.1176


              precision    recall  f1-score   support

       allow      0.097     1.000     0.177      2457
        warn      0.836     0.170     0.283     19055
       block      0.555     0.126     0.205      9868

    accuracy                          0.221     31380
   macro avg      0.496     0.432     0.222     31380
weighted avg      0.690     0.221     0.250     31380



Predicted decision,allow,warn,block
True decision,,,
allow,2457,0,0
warn,14815,3247,993
block,7994,635,1239


## Regex Baseline Results

The Regex baseline evaluated **31,380 validation prompts**. For each prompt, the project policy gives an **expected decision**, and the Regex baseline gives a **predicted decision**.

- `allow` means let the prompt through.
- `warn` means show a warning.
- `block` means stop the prompt.

The three tables show the same evaluation from different angles. The bottom table shows the individual counts, the middle table turns those counts into scores, and the top table shows how the scores vary by language.

### 1. Confusion matrix — what happened to each prompt?

Start with the **bottom table**. Each **row** shows what should have happened. Each **column** shows what the Regex baseline actually predicted. The **bold numbers on the diagonal** are the correct decisions.

| Expected decision ↓ / Predicted decision → | `allow` | `warn` | `block` | Total expected |
|---|---:|---:|---:|---:|
| `allow` | **2,457** | 0 | 0 | 2,457 |
| `warn` | 14,815 | **3,247** | 993 | 19,055 |
| `block` | 7,994 | 635 | **1,239** | 9,868 |
| **Total predicted** | **25,266** | **3,882** | **2,232** | **31,380** |

Take the number **7,994** in the `block` row and the `allow` column. This means **7,994 prompts should have been blocked, but the baseline let them through**. Another **635** should have been blocked but received a warning. Together, these are **8,629 missed blocks**.

You can also read **down** a column. In the `block` column, the baseline made **2,232 block decisions** in total. Of those, **1,239 were correct** and **993 were false blocks**. Those 993 prompts should have received a warning.

### 2. Classification report — what do precision, recall, F1, and support mean?

| Metric | Question it answers | Example using `block` |
|---|---|---|
| **Support** | How many prompts should get this decision? | **9,868** prompts should be blocked. This is the `block` row total. |
| **Recall** | Of the prompts that should get this decision, how many got it? | **1,239 ÷ 9,868 = 12.6%**. Divide by the row total. |
| **Precision** | Of the prompts that got this decision, how many should have got it? | **1,239 ÷ 2,232 = 55.5%**. Divide by the column total. |
| **F1 score** | How well does the baseline do on both precision and recall? | The `block` F1 is **0.205**. A low value for either precision or recall pulls F1 down. |

Here are the scores from the middle table, with precision and recall shown as percentages.

| Decision | Support | Precision | Recall | F1 |
|---|---:|---:|---:|---:|
| `allow` | 2,457 | 9.7% | 100.0% | 0.177 |
| `warn` | 19,055 | 83.6% | 17.0% | 0.283 |
| `block` | 9,868 | 55.5% | 12.6% | 0.205 |

- **`allow` has 100% recall because all 2,457 prompts that should have been allowed were allowed.** Its precision is only **9.7%** because the baseline allowed **25,266** prompts in total, including many that should have received a warning or a block.
- **`warn` has 83.6% precision.** When the baseline warned, it was usually right. Its recall is **17.0%** because it found only **3,247 of the 19,055** prompts that should have received a warning.
- **`block` has 12.6% recall.** For every 100 prompts that should have been blocked, the baseline blocked about 13. This is the score to look at when asking how often a required block was missed.

### 3. Summary scores — how well did the baseline do overall?

- **Accuracy is 22.1%.** Add the three correct counts on the diagonal to get **6,943 correct decisions**. Divide that by **31,380 prompts**.
- **Macro average gives each class equal weight.** For Macro-F1, take the F1 scores for `allow`, `warn`, and `block` and average them. The result is **0.2217**, displayed as **0.222** in the middle table.
- **Weighted average gives more weight to classes with more prompts.** `warn` has **19,055** prompts, so it has the largest influence. The weighted F1 is **0.250**. The same averaging idea applies to the precision and recall columns.

### 4. Results by language — how do I read the top table?

`overall` covers all four languages together. `prompts` is the number of texts evaluated in that row. The three recall columns each use the number of prompts that **should have received that specific decision**.

| Language | Prompts | Macro-F1 | Recall `allow` | Recall `warn` | Recall `block` |
|---|---:|---:|---:|---:|---:|
| All languages | 31,380 | 0.2217 | 100% | 17.04% | 12.56% |
| German (`de`) | 7,922 | 0.2215 | 100% | 16.65% | 13.06% |
| English (`en`) | 6,522 | 0.2288 | 100% | 17.59% | 12.89% |
| French (`fr`) | 9,292 | 0.2212 | 100% | 17.20% | 12.54% |
| Italian (`it`) | 7,644 | 0.2165 | 100% | 16.78% | 11.76% |

For example, the German row contains **7,922 prompts in total**. The **13.06% block recall** is calculated using only the German prompts that should have been blocked. It means the baseline blocked about **13 out of every 100 German block cases**.

### What does this mean for PromptGuard?

The most important weakness is **missed blocks**. One reason is that the Regex baseline searches for five specific types of sensitive information, while the policy covers additional types such as passwords. A prompt containing both an email address and a password can therefore receive `warn` when it should receive `block`. The baseline may have found the email correctly, but the decision for the **whole prompt** was still wrong.

## 6. Evaluate the detected text spans

A true positive needs the same start offset, end offset and label as the annotation.
A partial match counts as one false positive and one false negative. This table covers the
five types targeted by the regex rules. The prompt metrics above cover all policy types.

In [7]:
def span_keys(mask):
    """Turn a list of spans into a set of (start, end, label) tuples.

    Sets make the comparison easy: gold & predicted = found in both, gold - predicted
    = only annotated, and so on. Only the five labels that the regex rules can predict
    are kept; the other dataset labels (names, passwords, ...) are ignored here.
    """
    if isinstance(mask, str):     # depending on how the data was stored
        mask = json.loads(mask)
    return {
        (int(span["start"]), int(span["end"]), span["label"])
        for span in mask
        if span["label"] in RULE_LABELS
    }


def span_metrics(frame):
    """Exact-match precision, recall and F1 per label over all prompts in frame."""
    # One counter per outcome; each maps label -> count
    tp, fp, fn = Counter(), Counter(), Counter()
    for gold_mask, predicted_mask in zip(frame["privacy_mask"], frame["predicted_spans"]):
        gold = span_keys(gold_mask)            # annotated spans
        predicted = span_keys(predicted_mask)  # spans found by the regex rules
        tp.update(label for _, _, label in gold & predicted)   # in both: correct
        fp.update(label for _, _, label in predicted - gold)   # only predicted: false alarm
        fn.update(label for _, _, label in gold - predicted)   # only annotated: missed
        # A span with wrong boundaries is not in the intersection, so it counts
        # as one FP (the predicted span) AND one FN (the annotated span).

    rows = []
    for label in RULE_LABELS:
        correct, extra, missed = tp[label], fp[label], fn[label]
        support = correct + missed        # number of annotated spans of this label
        n_predicted = correct + extra     # number of predicted spans of this label
        rows.append({
            "label": label,
            "support": support,
            "TP": correct,
            "FP": extra,
            "FN": missed,
            # Precision: share of predicted spans that are correct
            "precision": correct / n_predicted if n_predicted else 0.0,
            # Recall: share of annotated spans that were found
            "recall": correct / support if support else 0.0,
            # F1 = 2·TP / (2·TP + FP + FN), written here with the two totals
            "F1": 2 * correct / (n_predicted + support) if n_predicted + support else 0.0,
        })
    return pd.DataFrame(rows).set_index("label")


# Span metrics for all languages and for German only, shown in one table
span_overall = span_metrics(validation)
span_de = span_metrics(validation.loc[validation["language"] == "de"])
display(pd.concat({"overall": span_overall, "de": span_de}, names=["scope"]).round(4))

support    TP    FP    FN  precision  recall      F1
scope   label                                                                 
overall EMAIL                2705  2683    26    22     0.9904  0.9919  0.9911
        IBAN                 1511   212     0  1299     1.0000  0.1403  0.2461
        ACCOUNTNUMBER        1885   459    19  1426     0.9603  0.2435  0.3885
        CREDITCARDNUMBER     1936   167  1406  1769     0.1062  0.0863  0.0952
        PHONENUMBER          1853  1563   136   290     0.9200  0.8435  0.8801
de      EMAIL                 652   645     5     7     0.9923  0.9893  0.9908
        IBAN                  369    63     0   306     1.0000  0.1707  0.2917
        ACCOUNTNUMBER         487   112     6   375     0.9492  0.2300  0.3702
        CREDITCARDNUMBER      473    44   376   429     0.1048  0.0930  0.0985
        PHONENUMBER           492   411    34    81     0.9236  0.8354  0.8773

## How well does Regex find sensitive information?

This table looks at **individual pieces of sensitive information**, such as an email address or an IBAN. One prompt can contain several of them.

A match is correct when Regex finds **the right type and the exact text**. For example, if it finds only part of a phone number, the partial match counts as a false positive and the full number counts as missed.

- `overall` includes all four languages.
- `de` shows only the German prompts.

### What do the columns mean?

| Column | Simple meaning |
|---|---|
| **Support** | How many examples of this type were marked in the validation data? |
| **TP** | How many did Regex find correctly? |
| **FP** | How many matches did Regex report incorrectly? |
| **FN** | How many marked examples did Regex miss? |
| **Precision** | When Regex reports a match, how often is it correct? |
| **Recall** | Of all the marked examples, how many does Regex find? |
| **F1** | A score from 0 to 1 that combines precision and recall. Higher is better. |

### What do the results show?

- **Email works best.** Regex found **2,683 out of 2,705** marked email addresses. It missed **22** and reported **26** incorrect matches. Both precision and recall are about **99%**.
- **Phone numbers work well.** Regex found **1,563 out of 1,853** marked phone numbers. That is **84.4% recall**.
- **IBAN matches are accurate but rare.** All **212** reported IBAN matches were correct, so precision is **100%**. There were **1,511** marked IBANs in total, so Regex missed **1,299**. That is only **14.0% recall**.
- **Account numbers show a similar pattern.** About **96%** of reported matches were correct, but Regex found only **459 out of 1,885** marked account numbers. That is **24.4% recall**.
- **Credit card numbers are the weakest result.** Regex correctly found **167 out of 1,936** marked card numbers and reported **1,406** incorrect matches. It both missed many card numbers and produced many false alarms.

### What about German prompts?

The German rows show a similar pattern. Regex found **645 of 652** email addresses and **411 of 492** phone numbers. For IBANs, it found **63 of 369**. All 63 matches were correct, but it still missed **306**.

### How does this relate to `allow`, `warn`, and `block`?

This table tells you **what Regex found in the text**. The previous table tells you **what action it chose for the whole prompt**. For example, Regex can correctly find an email address and choose `warn`, even though a password elsewhere in the same prompt means the correct action was `block`.

## 7. Inspect missed blocks and false alarms

The first table counts block-labelled annotations in prompts the baseline missed. A prompt
can contain several labels. The second table counts the labels that triggered a block in
prompts whose reference decision was allow or warn.

In [8]:
# Missed blocks: should be blocked, but the baseline said allow or warn (data leak).
# False blocks: should be allow/warn, but the baseline blocked (annoys the user).
missed_blocks = validation.loc[(actual == "block") & (predicted != "block")]
false_blocks = validation.loc[(actual != "block") & (predicted == "block")]
print(f"Missed blocks: {len(missed_blocks):,} of {(actual == 'block').sum():,} true block prompts")
print(f"False blocks: {len(false_blocks):,} of {(actual != 'block').sum():,} true allow/warn prompts")


def count_block_labels(masks):
    """Count how often each block-level label occurs in a column of span lists."""
    counts = Counter()
    for mask in masks:
        if isinstance(mask, str):
            mask = json.loads(mask)
        # Only labels whose policy decision is "block"
        counts.update(
            span["label"] for span in mask
            if policy["label_to_decision"].get(span["label"]) == "block"
        )
    # As a Series, most frequent label first
    return pd.Series(dict(counts), dtype="int64").sort_values(ascending=False)


# Annotated block labels in the missed prompts -> shows which PII types the rules miss
display(count_block_labels(missed_blocks["privacy_mask"]).to_frame("annotations in missed blocks"))
# Predicted block labels in the false blocks -> shows which rule caused the false alarm
display(count_block_labels(false_blocks["predicted_spans"]).to_frame("matches in false blocks"))

Missed blocks: 8,629 of 9,868 true block prompts
False blocks: 993 of 21,512 true allow/warn prompts


,annotations in missed blocks
PASSWORD,1812
CREDITCARDNUMBER,1699
SSN,1484
ACCOUNTNUMBER,1382
MASKEDNUMBER,1361
IBAN,1251
CREDITCARDCVV,552
PIN,448


,matches in false blocks
CREDITCARDNUMBER,996


## Why did the Regex baseline make mistakes?

This section looks at two types of errors: **missed blocks** and **false blocks**.

| Error | What it means | Result |
|---|---|---|
| **Missed block** | The prompt should have been blocked, but Regex returned `allow` or `warn`. | **8,629 out of 9,868** prompts that should have been blocked (**87.4%**). |
| **False block** | The prompt should have received `allow` or `warn`, but Regex blocked it. | **993 out of 21,512** prompts that should have received `allow` or `warn` (**4.6%**). |

### What was in the missed prompts?

The first table counts sensitive items **marked in the dataset**. For example, `PASSWORD = 1,812` means the missed prompts contained **1,812 marked password entries**. One prompt can contain several sensitive items, so the table counts **items**, while the error summary counts **prompts**.

Passwords, SSNs, and PINs appear in the table and are outside the five Regex rules. The table also shows **1,699 credit card number entries**, **1,382 account number entries**, and **1,251 IBAN entries** in missed prompts. Regex has rules for these three types, but those rules still missed many cases.

### What triggered the false blocks?

The second table counts what **Regex reported** in prompts that should have received `allow` or `warn`. All **996 block-triggering matches** came from the credit card rule. There were **993 prompts** because some prompts contained more than one match.

The credit card rule can accept a number that looks like a card number and passes the Luhn check. Other kinds of numbers can also pass that check, which helps explain how a false block can happen.

## 8. Measure response time

Time one prompt at a time, including rule matching and the policy decision. Use the same
sample size and seed as the TF-IDF notebook. Compare latency on the same machine.

In [9]:
# Random sample of up to 1,000 validation prompts. Same size and seed as in the
# TF-IDF notebook, so both models are timed on the same prompts.
sample = validation["source_text"].sample(min(1000, len(validation)), random_state=SEED)
times_ms = []
for prompt in sample:
    start = time.perf_counter()          # high-resolution timer (seconds)
    predict_prompt(prompt)               # full pipeline: detection + policy decision
    times_ms.append((time.perf_counter() - start) * 1000)   # seconds -> milliseconds

# Median = time of a typical prompt.
# 95th percentile = 95 % of the prompts are faster than this (shows slow outliers).
LATENCY_MEDIAN_MS = float(np.median(times_ms))
LATENCY_P95_MS = float(np.percentile(times_ms, 95))
print(f"Median: {LATENCY_MEDIAN_MS:.3f} ms per prompt")
print(f"95th percentile: {LATENCY_P95_MS:.3f} ms per prompt")

Median: 0.134 ms per prompt
95th percentile: 0.338 ms per prompt


## 9. Log the validation run to MLflow

The local promptguard experiment uses the same metric names and tracking database as
the TF-IDF baseline. The notebook contains the detector. MLflow records the rule version,
policy, split, metrics and the validation tables.

In [10]:
import mlflow

# Same local tracking database (mlflow.db in the project folder) as the TF-IDF
# baseline, so both runs appear side by side in the MLflow UI.
mlflow.set_tracking_uri(f"sqlite:///{(PROJECT_DIR / 'mlflow.db').as_posix()}")
# Create the experiment only once; files of the runs (artifacts) are saved in mlruns/
if mlflow.get_experiment_by_name("promptguard") is None:
    mlflow.create_experiment("promptguard", artifact_location=(PROJECT_DIR / "mlruns").as_uri())
mlflow.set_experiment("promptguard")

with mlflow.start_run(run_name="regex_baseline"):
    # Parameters: WHAT was evaluated (model type, rule version, labels, split, policy)
    mlflow.log_params({
        "model": "regex",
        "ruleset_version": RULESET_VERSION,
        "labels": ", ".join(RULE_LABELS),
        "split_fingerprint": SPLIT_FINGERPRINT,
        "policy_version": str(POLICY_VERSION),
        "policy_sha256": POLICY_SHA256,
    })
    # Metrics: same names as in the TF-IDF notebook, so the runs can be compared directly
    mlflow.log_metrics({
        **{f"val_{metric}": value for metric, value in overall.items()},
        "val_de_macro_f1": by_language["de"]["macro_f1"],
        "val_de_recall_block": by_language["de"]["recall_block"],
        "latency_median_ms": LATENCY_MEDIAN_MS,
        "latency_p95_ms": LATENCY_P95_MS,
    })
    # The full result tables as one JSON file in the run's artifacts.
    # to_json() + json.loads() converts the DataFrames into plain dicts.
    mlflow.log_dict(
        {
            "decision_metrics": json.loads(decisions.to_json(orient="index")),
            "confusion_matrix": json.loads(confusion.to_json(orient="index")),
            "exact_spans_overall": json.loads(span_overall.to_json(orient="index")),
            "exact_spans_de": json.loads(span_de.to_json(orient="index")),
        },
        "validation_metrics.json",
    )
print("Regex baseline logged to MLflow.")

c:\Users\annis\MWI_AIOps\AIOps_PromptGuard\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Regex baseline logged to MLflow.


## 10. Interpret the results

- Compare validation **Macro F1** and **Recall block** with the TF-IDF baseline for all languages and for German.
- Use the span table to see which structured PII types the rules can locate. A correctly located e-mail can still yield a missed block if the same prompt contains a password outside the rules.
- Names, addresses, passwords, PINs and obfuscated values are outside the baseline's coverage. Unlabelled account numbers and some phone formats can also be missed. Checksum rules can miss annotated values that fail the format check.
- Keep the team's final test evaluation and separate realistic German test set for the final model comparison.
